In [3]:
from nlp import list_datasets

/Users/hyeon/Oracle/3_AI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
datasets_list = list_datasets()
print('\n '.join(dataset.id for dataset in datasets_list))

markov-ai/gaming-500-hours
 LiquidAI/antidoom-mix-v1.0
 FlyRank/internship-warehouse
 Glint-Research/Fable-5-traces
 ByteDance-Seed/EdgeBench
 LiquidAI/ifstruct-v1.0
 Crownelius/Complete-FABLE.5-traces-2M
 netflix/Vera-Layered-Video-Dataset
 kyutai/rocket-science
 CMRobot/MotionDecode
 armand0e/claude-fable-5-claude-code
 openai/gsm8k
 bigfacing/GOKU-2M
 AlicanKiraz0/Turkce-Atlas-Instruct
 sensenova/SenseNova-Vision-Corpus-50M
 ProCreations/grug-think
 HuggingFaceFW/fineweb-edu
 scholarweave/arxiv-latex
 AletheiaResearch/GLM-5.2-Agent
 Anthropic/hh-rlhf
 Qwen/AgentWorldBench
 RekaAI/CS2-10k
 wikimedia/wikipedia
 bones-studio/seed
 ASLP-lab/WSC-Train
 Rapidata/psychology-association-kiki-bouba-etc
 WithinUsAI/claude_mythos_distilled_25k
 SupraLabs/reasoning-summaries-61k
 yatin-superintelligence/blood-pathology-lims-environment
 yatin-superintelligence/digital-hospital-environment
 HuggingFaceFW/fineweb
 mlabonne/open-perfectblend
 Syn4D/Syn4D
 sarulab-speech/DuplexChat
 ministere-cultu

In [5]:
from transformers import BertModel, BertTokenizer
model = BertModel.from_pretrained("bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31344.27it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
tokenizer.tokenize("today is monday")

['today', 'is', 'monday']

In [7]:
tokens_result = tokenizer.tokenize("today is monday")


In [8]:
tokens = ['[CLS]']   +   tokens_result  + ['[SEP]']  

tokens = tokens + ["[PAD]"] + ["[PAD]"]


In [9]:
attention_mask = []
for i in tokens:
    if i != '[PAD]':
        attention_mask.append(1)
    else:
        attention_mask.append(0)
    # print(i)


In [10]:
attention_mask = [1 if i != '[PAD]' else 0 for i in tokens]
token_ids = tokenizer.convert_tokens_to_ids(tokens)

In [11]:
token_ids

[101, 2651, 2003, 6928, 102, 0, 0]

In [12]:
import torch


token_ids = torch.tensor(token_ids).to('mps')
attention_mask= torch.tensor(attention_mask).to('mps')


In [13]:
token_ids.shape

torch.Size([7])

In [14]:
token_ids = token_ids.unsqueeze(0)
attention_mask = attention_mask.unsqueeze(0)

In [15]:
model = model.to('mps')

In [16]:
outputs = model(token_ids, attention_mask=attention_mask)

In [17]:
outputs.last_hidden_state.shape

torch.Size([1, 7, 768])

In [47]:
text = "The capital of Korea is [MASK]."

In [48]:
inputs = tokenizer(text, return_tensors='pt')

In [49]:
inputs

{'input_ids': tensor([[ 101, 1996, 3007, 1997, 4420, 2003,  103, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [50]:
encoded = tokenizer("I Love NLP.", "BERT is powerful", return_tensors='pt')
encoded

{'input_ids': tensor([[  101,  1045,  2293, 17953,  2361,  1012,   102, 14324,  2003,  3928,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [51]:
inputs = inputs.to('mps')

In [52]:
with torch.no_grad():
    outputs = model(**inputs)

In [53]:
from transformers import BertForMaskedLM, BertForSequenceClassification

In [54]:
model = BertForMaskedLM.from_pretrained("bert-base-uncased")
model = model.to('mps')

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 16182.47it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
with torch.no_grad():
    outputs = model(**inputs)
    predictions = outputs.logits

In [56]:
mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)

In [57]:
inputs['input_ids']

tensor([[ 101, 1996, 3007, 1997, 4420, 2003,  103, 1012,  102]],
       device='mps:0')

In [65]:
pred = predictions[0, mask_token_index].argmax(dim=-1)

In [66]:
tokenizer.decode(pred)

'seoul'

In [75]:
text = "sehyeon looks like a [MASK]."
inputs = tokenizer(text, return_tensors="pt").to('mps')
with torch.no_grad():
    outputs = model(**inputs)
    predictions  = outputs.logits


mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
predicted_token_id = predictions[0, mask_token_index].argmax(dim=-1)
predicted_word = tokenizer.decode(predicted_token_id)

predicted_word


'man'

In [86]:
"The capital of Japan is " + tokenizer.decode(1238)

'The capital of Japan is ւ'

In [94]:
kmodel = BertForMaskedLM.from_pretrained('bert-base-multilingual-cased')
ktokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 16837.56it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [147]:
kname = "snunlp/KR-BERT-char16424"
kmodel = BertForMaskedLM.from_pretrained(kname)
ktokenizer = BertTokenizer.from_pretrained(kname)


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 87995.42it/s]
[transformers] BertForMaskedLM LOAD REPORT from: snunlp/KR-BERT-char16424
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [148]:
kmodel = kmodel.to('mps')

In [107]:
text = "주식 투자는 정해진 [MASK] 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다."


inputs = ktokenizer(text, return_tensors="pt").to('mps')
with torch.no_grad():
    outputs = kmodel(**inputs)
    predictions  = outputs.logits


In [114]:
predicted_token_id = predictions[0, mask_token_index].argmax(dim=-1)
predicted_word = ktokenizer.decode(predicted_token_id)


In [115]:
mask_token_index = torch.where(inputs["input_ids"] == ktokenizer.mask_token_id)[1]


predicted_word = ktokenizer.decode(predicted_token_id)


In [116]:
predicted_word

'##았다'

In [111]:
values , indices  = torch.topk(predictions[0, mask_token_index], k=10, dim=-1)




for x in indices[0]:
    print(f"주식 투자는 정해진 {ktokenizer.decode(x)} 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.")


주식 투자는 정해진 식 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 정 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 양 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 ( 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 : 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 , 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 연속 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 ##형 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 x 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.
주식 투자는 정해진 은 으로 어떤 주식을 얼마큼 들고 있을지에 대한 선택의 연속이다.


In [182]:
text = "정세현과 닮은 동물은 [MASK]이다."

inputs = ktokenizer(text, return_tensors='pt').to('mps')
with torch.no_grad():
    outputs = kmodel(**inputs)
    predictions  = outputs.logits


In [183]:
predicted_token_id = predictions[0, mask_token_index].argmax(dim=-1)
predicted_word = ktokenizer.decode(predicted_token_id)


In [184]:
mask_token_index = torch.where(inputs["input_ids"] == ktokenizer.mask_token_id)[1]


predicted_word = ktokenizer.decode(predicted_token_id)


In [185]:
values , indices  = torch.topk(predictions[0, mask_token_index], k=10, dim=-1)

In [186]:
for x in indices[0]:
    print(f"정세현과 닮은 동물은 {ktokenizer.decode(x)}이다.")

정세현과 닮은 동물은 ?이다.
정세현과 닮은 동물은 바로이다.
정세현과 닮은 동물은 [UNK]이다.
정세현과 닮은 동물은 )이다.
정세현과 닮은 동물은 ,이다.
정세현과 닮은 동물은 "이다.
정세현과 닮은 동물은 다음이다.
정세현과 닮은 동물은 다음과이다.
정세현과 닮은 동물은 유재석이다.
정세현과 닮은 동물은 사람이다.


In [151]:
predicted_token_id = predictions[0, mask_token_index].argmax(dim=-1)
predicted_word = ktokenizer.decode(predicted_token_id)


In [152]:
mask_token_index = torch.where(inputs["input_ids"] == ktokenizer.mask_token_id)[1]


predicted_word = ktokenizer.decode(predicted_token_id)


In [153]:
predicted_word

'##은'

In [204]:
text = "정세현은 낙타와 [MASK]다."
inputs = ktokenizer(text, return_tensors="pt").to('mps')
kmodel = kmodel.to('mps')
with torch.no_grad():
    outputs = kmodel(**inputs)
    predictions  = outputs.logits




mask_token_index = torch.where(inputs["input_ids"] == ktokenizer.mask_token_id)[1]
mask_logits = predictions[0, mask_token_index]




import torch.nn.functional as F
probs = F.softmax(mask_logits, dim=-1)


In [205]:
probs.sum()

tensor(1., device='mps:0')

In [206]:
top_values, top_indices = torch.topk(probs, k=20, dim=-1)

In [207]:
print(text)
for i in range(20):
    token_id = top_indices[0, i].item()
    token_prob = top_values[0, i].item()
    token_text = ktokenizer.decode([token_id])
   
    rank = i + 1
    print(f"{rank}순위: {token_text} (ID: {token_id}, 확률: {token_prob:.4f})")


정세현은 낙타와 [MASK]다.
1순위: 함께 (ID: 168, 확률: 0.1861)
2순위: 같은 (ID: 207, 확률: 0.1001)
3순위: 비슷한 (ID: 2700, 확률: 0.0799)
4순위: 둘 (ID: 2385, 확률: 0.0497)
5순위: [UNK] (ID: 1, 확률: 0.0322)
6순위: 돼지 (ID: 7042, 확률: 0.0279)
7순위: 같이 (ID: 1118, 확률: 0.0250)
8순위: 달리 (ID: 1222, 확률: 0.0220)
9순위: 나란히 (ID: 5540, 확률: 0.0179)
10순위: 비슷 (ID: 5799, 확률: 0.0107)
11순위: 하나 (ID: 598, 확률: 0.0101)
12순위: ' (ID: 12, 확률: 0.0099)
13순위: " (ID: 11, 확률: 0.0094)
14순위: ##도 (ID: 21, 확률: 0.0084)
15순위: 슬라이더 (ID: 7636, 확률: 0.0084)
16순위: 새 (ID: 581, 확률: 0.0081)
17순위: 나무 (ID: 3679, 확률: 0.0079)
18순위: 마찬가지 (ID: 5849, 확률: 0.0077)
19순위: 배 (ID: 353, 확률: 0.0070)
20순위: 다리 (ID: 3791, 확률: 0.0065)
